# How do we know if a model is good?
**Topics:** Train/Val/Test Splits · Cross-Validation · Bias-Variance Tradeoff · Overfitting/Underfitting · Regularization (L1/L2 · Early Stopping · Dropout)


---
## 1. Train / Validation / Test Splits

### What it is
- Divide data into 3 subsets to train, tune, and evaluate the model
- **Train** → model learns from this
- **Validation** → tune hyperparameters, monitor overfitting
- **Test** → final unbiased evaluation — touch only once

### Key intuition
- Test set = exam the model has never seen
- Using test set during tuning = cheating — inflated performance estimate

### Typical splits
- Small data (<10k): 60/20/20
- Large data (>100k): 98/1/1 is fine

### When to use
- Always — no exceptions
- Stratify splits for imbalanced classification (preserves class ratio)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
import numpy as np

X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

# Split into train+val and test first
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Then split train+val into train and val
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)
# Result: 60% train, 20% val, 20% test

print(f"Train : {len(X_train)}")
print(f"Val   : {len(X_val)}")
print(f"Test  : {len(X_test)}")


### Interview Q&A

**Why not just use train/test?**
- You need a separate validation set to tune hyperparameters without contaminating the test set

**What is data leakage?**
- When information from val/test set leaks into training → artificially inflated performance
- Common cause: fitting scaler on full dataset before splitting → always fit on train only

### Gotchas
- Fit preprocessors (scalers, encoders) on train set only — transform val/test separately
- Shuffle before splitting unless data is time-series (then use time-based split)


---
## 2. Cross-Validation

### What it is
- Technique to evaluate model performance more reliably using multiple train/val splits
- K-Fold: split data into K folds, train on K-1, validate on 1, rotate K times
- Final score = mean across all K folds

### Key intuition
- Single train/val split is noisy — depends on which samples end up in val
- Cross-validation averages out this noise → more reliable estimate

### When to use
- Small to medium datasets where you can't afford a fixed val set
- Comparing models or hyperparameter settings reliably

### When not to use
- Very large datasets — too computationally expensive
- Time-series data → use TimeSeriesSplit instead (preserves temporal order)


In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
import numpy as np

X, y = make_classification(n_samples=500, n_features=10, random_state=42)

model = LogisticRegression()

# Standard K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')
print(f"K-Fold CV scores : {scores.round(3)}")
print(f"Mean ± Std       : {scores.mean():.3f} ± {scores.std():.3f}")

# Stratified K-Fold (for classification — preserves class ratio)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_s = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
print(f"Stratified scores: {scores_s.round(3)}")
print(f"Mean ± Std       : {scores_s.mean():.3f} ± {scores_s.std():.3f}")


### Interview Q&A

**K-Fold vs Stratified K-Fold?**
- K-Fold → random splits, good for regression
- Stratified K-Fold → preserves class ratio per fold, better for classification

**What K to choose?**
- K=5 or K=10 are standard — good bias/variance tradeoff
- K=N (leave-one-out) → most accurate but very expensive

### Gotchas
- Apply preprocessing inside each fold, not before — otherwise data leakage
- Use `Pipeline` with `cross_val_score` to safely include preprocessing


---
## 3. Bias-Variance Tradeoff

### What it is
- **Bias** → error from wrong assumptions — model too simple, underfits
- **Variance** → error from sensitivity to training data — model too complex, overfits
- Total error = Bias² + Variance + Irreducible noise

### Key intuition
- High bias → model misses the pattern (straight line on curved data)
- High variance → model memorizes noise (wiggly curve that fits every training point)
- Goal → find the sweet spot in between

### Signals

| | Training error | Validation error |
|---|---|---|
| High bias (underfit) | High | High |
| High variance (overfit) | Low | High |
| Good fit | Low | Low (close to train) |

### When to use this framework
- Diagnosing why a model performs poorly
- Deciding whether to add complexity or simplify


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

np.random.seed(42)
X = np.sort(np.random.rand(80, 1) * 6 - 3, axis=0)
y = np.sin(X).ravel() + np.random.randn(80) * 0.3

train_errors, val_errors = [], []
degrees = range(1, 12)

for d in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d)),
        ('lr',   LinearRegression())
    ])
    train_score = cross_val_score(pipe, X, y, cv=5, scoring='neg_mean_squared_error',
                                   error_score='raise')
    train_errors.append(-train_score.mean())

    val_score = cross_val_score(pipe, X, y, cv=5, scoring='neg_mean_squared_error')
    val_errors.append(-val_score.mean())

print("Degree | Train MSE | Val MSE")
for d, tr, vl in zip(degrees, train_errors, val_errors):
    print(f"  {d:2d}   |  {tr:.4f}   | {vl:.4f}")
# Low degree → high bias (both errors high)
# High degree → high variance (train low, val high)


### Interview Q&A

**How do you fix high bias?**
- Use a more complex model, add more features, reduce regularization

**How do you fix high variance?**
- Get more training data, simplify model, increase regularization, use dropout

**Can you have both high bias and high variance?**
- Yes — e.g. a decision tree on a noisy irrelevant feature set

### Gotchas
- Always plot learning curves — they reveal bias/variance problems visually
- More data fixes variance but not bias


---
## 4. Overfitting & Underfitting

### What it is
- **Overfitting** → model learns training data too well including noise → poor generalization
- **Underfitting** → model too simple to capture the underlying pattern

### Key intuition
- Overfit: memorizes exam answers but fails new questions
- Underfit: hasn't studied enough — gets everything wrong

### How to detect

**Overfitting:**
- Training accuracy >> validation accuracy
- Validation loss increases while training loss decreases

**Underfitting:**
- Both training and validation accuracy are low
- Model performs near random baseline

### When to use this framework
- Any time model performs differently on train vs val/test


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

print(f"{'Depth':<8} {'Train Acc':<12} {'Test Acc':<12} {'Diagnosis'}")
print("-" * 50)
for depth in [1, 2, 5, 10, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    tr = accuracy_score(y_train, model.predict(X_train))
    te = accuracy_score(y_test,  model.predict(X_test))
    diff = tr - te
    diagnosis = "Underfit" if tr < 0.75 else ("Overfit" if diff > 0.1 else "Good fit")
    print(f"{str(depth):<8} {tr:<12.3f} {te:<12.3f} {diagnosis}")


### Interview Q&A

**3 ways to reduce overfitting?**
- More training data, regularization (L1/L2), simpler model / fewer features

**What is early stopping?**
- Stop training when validation loss stops improving → prevents overfitting in iterative models

### Gotchas
- A perfect training accuracy (1.0) is almost always a sign of overfitting
- Check if test set was accidentally used during development — inflated test results


---
## 5. Regularization

### What it is
- Techniques that penalize model complexity to reduce overfitting
- Adds a penalty term to the loss function that discourages large weights

### L1 vs L2

| | L1 (Lasso) | L2 (Ridge) |
|---|---|---|
| Penalty | Sum of \|weights\| | Sum of weights² |
| Effect | Drives some weights to exactly 0 | Shrinks all weights toward 0 |
| Use for | Feature selection | When all features matter |
| Result | Sparse model | Small but non-zero weights |

### When to use
- L1 → many irrelevant features, want automatic feature selection
- L2 → features are correlated, want to keep all but shrink them
- ElasticNet → combine both L1 and L2

### Dropout (neural networks)
- Randomly sets fraction of neurons to 0 during each training step
- Forces network to learn redundant representations → more robust
- Only applied during training, not inference


In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import numpy as np

X, y = make_regression(n_samples=200, n_features=20, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

models = {
    'Ridge (L2)'    : Ridge(alpha=1.0),
    'Lasso (L1)'    : Lasso(alpha=0.1),
    'ElasticNet'    : ElasticNet(alpha=0.1, l1_ratio=0.5),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    mse = mean_squared_error(y_test, model.predict(X_test))
    n_zero = np.sum(np.abs(model.coef_) < 1e-4)
    print(f"{name:<18} MSE: {mse:8.2f}  Zero weights: {n_zero}")

# Dropout example (PyTorch)
# import torch.nn as nn
# nn.Dropout(p=0.5)   # 50% of neurons zeroed during training


### Interview Q&A

**Why does L1 produce sparse weights but L2 doesn't?**
- L1 penalty has a sharp corner at 0 → gradient pushes weights exactly to 0
- L2 penalty is smooth → gradient approaches 0 asymptotically, never quite reaches it

**What is the alpha (λ) hyperparameter?**
- Controls regularization strength — higher alpha → stronger penalty → simpler model
- Tune with cross-validation

**When would you use dropout?**
- Deep neural networks with many parameters — not typically used in linear models

### Gotchas
- Regularization requires feature scaling — unscaled features make regularization unfair across features
- Too high alpha → underfitting (model too constrained)
- Dropout rate 0.2–0.5 is typical — too high and model can't learn
